# NB06 · Semana 14 — Serving: gold en PostgreSQL + Metabase

**Ingeniería de Datos — LCD · UCA Rosario · 2026**  
Versión: **para el alumno (a completar)**

## Objetivo

Cerrar el pipeline: **materializar la capa gold** en PostgreSQL (warehouse de serving) y construir un **tablero en Metabase** para usuarios de negocio.

> Base de la **Entrega 4 (E4)**.

## 1. Leer la gold desde DuckDB

La capa gold fue calculada por dbt (NB04). La leemos para publicarla.

In [2]:
import os, sys
import duckdb

sys.path.insert(0, '/home/jovyan/work')
from entorno import DBT_DIR

ruta = os.path.join(DBT_DIR, 'warehouse.duckdb')
con = duckdb.connect(ruta, read_only=True)
gold = con.execute(
    "select * from fct_trafico_snapshot order by snapshot_ts"
).df()
con.close()

print("Snapshots en la capa gold:", gold.shape)
gold.head()

Snapshots en la capa gold: (13, 7)


,snapshot_ts,pilotos_conectados,pilotos_sin_plan,aeropuertos_origen,aeropuertos_destino,pilotos_previo,var_pct_pilotos
0,2026-09-25 22:18:23.788091,1137,51,326,369,<NA>,NaN
1,2026-09-25 22:18:38.789216,1135,48,326,369,1137,-0.18
2,2026-09-25 22:35:23.787757,1108,38,325,366,1135,-2.38
3,2026-09-25 22:35:53.789119,1105,37,323,365,1108,-0.27
4,2026-09-25 22:38:08.788199,1103,40,327,366,1105,-0.18


## 2. Publicar en PostgreSQL

Escribimos la gold en la base `gold` de PostgreSQL, que Metabase consultará. Estrategia: reemplazo total (`replace`) en cada corrida.

⚠️ **Requiere el stack activo** (servicio postgres).

In [3]:
from sqlalchemy import create_engine, text
from entorno import pg_url

ESQUEMA, TABLA = "grupo03", "fct_trafico_snapshot"

motor = create_engine(pg_url())
with motor.begin() as con:
    gold.to_sql(TABLA, con, schema=ESQUEMA,
                if_exists="replace", index=False, chunksize=500)
    total = con.execute(
        text(f"select count(*) from {ESQUEMA}.{TABLA}")).scalar_one()
motor.dispose()

print(f"Publicadas {total} filas en warehouse.{ESQUEMA}.{TABLA}")
assert total == len(gold), "La cantidad publicada no coincide"

Publicadas 13 filas en warehouse.grupo03.fct_trafico_snapshot


## 3. Construir el tablero en Metabase

Esta parte es manual, en la UI de Metabase (http://localhost:3000):

1. En el primer uso, conectar Metabase a la base **gold** (host `postgres`, puerto `5432`, base `gold`, usuario `airflow`).
2. Crear preguntas sobre `fct_series_mensual`: evolución del valor promedio por mes, comparación entre variables, etc.
3. Agrupar las preguntas en un **dashboard**.

> **Metabase vs. Grafana:** Metabase responde preguntas de *negocio* sobre el warehouse (esta clase); Grafana se usa para *observabilidad* del pipeline (semana 13). Cada herramienta en su rol.

## Entrega 4 (E4)

Lo que se evalúa: capa gold en PostgreSQL, features reproducibles, tablero de negocio en Metabase y documentación de governance del dataset. Enunciado y rúbrica en la semana 13.